# 02 — Analysis & Insights

**Input**: `data/cleaned_jobs.csv`

Questions answered:
1. Most demanded IT skills
2. Top hiring cities
3. Remote / Hybrid / Onsite ratio
4. Salary by experience level
5. Skills by role (Intern vs Backend vs Frontend)
6. Salary disclosure rate
7. Work model distribution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

plt.style.use('ggplot')
sns.set_palette('Set2')
%matplotlib inline

In [ ]:
df = pd.read_csv('data/cleaned_jobs.csv')
print(f'Loaded {len(df)} cleaned job postings')
df.head(2)

## 1. Top 15 Most Demanded Skills

In [ ]:
all_skills = []
for skills_str in df['skills'].dropna():
    all_skills.extend(s for s in skills_str.split(';') if s.strip())

skill_counts = Counter(all_skills)
top15 = skill_counts.most_common(15)

fig, ax = plt.subplots(figsize=(10, 6))
names, counts = zip(*top15)
ax.barh(list(names)[::-1], list(counts)[::-1])
ax.set_xlabel('Job Count'); ax.set_title('Top 15 IT Skills in Demand')
plt.tight_layout(); plt.show()

## 2. Jobs by City

In [ ]:
city_counts = df['city'].value_counts().head(10)
fig, ax = plt.subplots(figsize=(8, 5))
city_counts.plot.bar(ax=ax)
ax.set_title('Top 10 Hiring Cities'); ax.set_ylabel('Job Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

## 3. Work Model Distribution

In [ ]:
wm = df['work_model'].value_counts()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
wm.plot.pie(ax=ax1, autopct='%1.1f%%', startangle=90)
ax1.set_ylabel('')
wm.plot.bar(ax=ax2)
ax2.set_title('Work Model Distribution')
ax2.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 4. Experience Level Distribution

In [ ]:
exp = df[df['experience_level'] != 'Unknown']['experience_level']
fig, ax = plt.subplots(figsize=(8, 4))
exp.value_counts().sort_index().plot.bar(ax=ax)
ax.set_title('Jobs by Experience Level'); ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## 5. Salary Range by Experience Level

In [ ]:
salary_df = df[df['is_salary_public'] & (df['experience_level'] != 'Unknown')].copy()
salary_df['avg_salary'] = (salary_df['salary_min'] + salary_df['salary_max']) / 2

fig, ax = plt.subplots(figsize=(8, 5))
order = ['Internship', 'Fresher', 'Junior', 'Mid-level', 'Senior', 'Manager']
sns.boxplot(data=salary_df, x='experience_level', y='avg_salary', order=order, ax=ax)
ax.set_title('Salary Range by Experience Level (Million VND/month)')
ax.set_xlabel(''); ax.set_ylabel('Average Salary (Million VND)')
plt.tight_layout(); plt.show()

## 6. Salary Disclosure Rate

In [ ]:
disclosed = df['is_salary_public'].sum()
total = len(df)
print(f'Salary disclosed: {disclosed}/{total} ({disclosed/total*100:.1f}%)')

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie([disclosed, total - disclosed], labels=['Disclosed', 'Not Disclosed'],
       autopct='%1.1f%%', startangle=90, colors=['#4CAF50', '#E0E0E0'])
ax.set_title('Salary Disclosure Rate')
plt.show()

## 7. Skills for Intern / Fresher vs Senior roles

What skills are most common at entry level vs senior level?

In [ ]:
def skills_by_level(df, level):
    subset = df[df['experience_level'] == level]
    all_s = []
    for s in subset['skills'].dropna():
        all_s.extend(x.strip() for x in s.split(';') if x.strip())
    return Counter(all_s).most_common(10)

entry_skills = skills_by_level(df, 'Internship') + skills_by_level(df, 'Fresher')
senior_skills = skills_by_level(df, 'Senior') + skills_by_level(df, 'Manager')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

if entry_skills:
    names, vals = zip(*entry_skills[:10])
    ax1.barh(list(names)[::-1], list(vals)[::-1])
ax1.set_title('Entry-level (Intern/Fresher)')

if senior_skills:
    names, vals = zip(*senior_skills[:10])
    ax2.barh(list(names)[::-1], list(vals)[::-1])
ax2.set_title('Senior/Manager')

plt.tight_layout(); plt.show()

## 8. Summary Statistics

In [ ]:
print('='*50)
print('Vietnam IT Job Market — Summary')
print('='*50)
print(f'Total jobs analyzed: {len(df)}')
print(f'Companies represented: {df["company_name"].nunique()}')
print(f'Cities represented: {df["city"].nunique()}')
print(f'Salary disclosure: {disclosed/total*100:.1f}%')
print(f'\nTop 5 cities:')
for city, cnt in df['city'].value_counts().head(5).items():
    print(f'  {city}: {cnt}')
print(f'\nWork model distribution:')
for wm, cnt in df['work_model'].value_counts().items():
    print(f'  {wm}: {cnt} ({cnt/total*100:.1f}%)')